<a href="https://colab.research.google.com/github/anamacao/FAPESP-PIBIC-scrapping/blob/main/Mercosul/dataprivacy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import time
import re
import plotly.express as px
import plotly.io as pio
import matplotlib.pyplot as plt
import seaborn as sns

pio.renderers.default = "notebook_connected"

In [ ]:
BASE_URL_TEMPLATE = "https://dataprivacy.com.br/category/noticias/page/{}/"
DATABASE_NAME = "dataprivacy.db"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

print("Configuração pronta!")

In [ ]:
def create_database():
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS articles (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            date TEXT,
            url TEXT UNIQUE,
            source TEXT
        )
    """)
    conn.commit()
    conn.close()
    print("✅ Banco e tabela prontos")

create_database()

In [ ]:
def insert_article(title, date, url, source="Data Privacy Brasil"):
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    try:
        cursor.execute("""
            INSERT INTO articles (title, date, url, source)
            VALUES (?, ?, ?, ?)
        """, (title, date, url, source))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

In [ ]:
inserted_count = 0
page = 1
max_failures = 3
failures = 0

# Set para detectar duplicatas e parar loop infinito
seen_urls = set()

while True:
    # URL: A primeira página é diferente ou igual?
    # O site pode ignorar /page/X/ e retornar sempre a home.
    if page == 1:
        url = "https://dataprivacy.com.br/category/noticias/"
    else:
        url = BASE_URL_TEMPLATE.format(page)

    print(f"Coletando página {page}: {url}")

    try:
        r = requests.get(url, headers=HEADERS, timeout=15)

        if r.status_code == 404:
            print("  Página não encontrada (404). Fim da raspagem.")
            break

        soup = BeautifulSoup(r.text, "html.parser")

        articles = soup.find_all("article")
        if not articles:
             articles = soup.select(".post, .entry, .card")

        print(f"  {len(articles)} notícias encontradas")

        if len(articles) == 0:
            print("  Nenhum artigo encontrado.")
            break

        # Verifica duplicidade na página inteira
        # Se todos os artigos desta página já foram vistos, estamos em loop
        page_new_urls = 0

        for article in articles:
            # Título
            title_tag = article.select_one("h2.entry-title, h3.entry-title, h2 a, h3 a")
            if not title_tag:
                 title_tag = article.find("h2") or article.find("h3")

            if not title_tag:
                continue

            title = title_tag.get_text(strip=True)

            # Link
            link_tag = article.find("a")
            if title_tag.name == 'a':
                link_tag = title_tag

            link = link_tag['href'] if link_tag else ""

            # Checa duplicata na sessão atual
            if link in seen_urls:
                continue

            seen_urls.add(link)
            page_new_urls += 1

            # Data
            date_tag = article.select_one("time, .date, .post-date, .entry-date, .published")
            date = ""
            if date_tag:
                date = date_tag.get_text(strip=True)

            if not date:
                text = article.get_text()
                match = re.search(r'\d{2}[./]\d{2}[./]\d{4}', text)
                if match:
                    date = match.group(0)

            if insert_article(title, date, link):
                inserted_count += 1

        # Se não achou NENHUM URL novo nesta página, aborta
        if page_new_urls == 0:
            print("⚠️ Todos os artigos desta página já foram lidos. Loop detectado ou fim da paginação real.")
            break

        page += 1
        time.sleep(1)

    except Exception as e:
        print(f"Erro na página {page}: {e}")
        break

print(f"\n📥 Total de novas notícias inseridas: {inserted_count}")

In [ ]:
def load_articles():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("SELECT * FROM articles", conn)
    conn.close()
    return df

df_db = load_articles()
print(f"📦 Total no banco: {len(df_db)} registros")
display(df_db.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import re

if not df_db.empty:
    # 1. Timeline Estática
    df_db['date_dt'] = pd.to_datetime(df_db['date'], dayfirst=True, errors='coerce')
    timeline_data = df_db.dropna(subset=['date_dt']).sort_values('date_dt')
    timeline_counts = timeline_data['date_dt'].value_counts().sort_index()

    plt.figure(figsize=(12, 5))
    plt.plot(timeline_counts.index, timeline_counts.values, marker='o', linestyle='-', color='teal')
    plt.title('Frequência de Notícias ao Longo do Tempo (Gráfico Estático)')
    plt.xlabel('Data de Publicação')
    plt.ylabel('Quantidade de Artigos')
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # 2. Informação textual complementar
    print(f"\nTotal de artigos processados para o gráfico: {len(timeline_data)}")
else:
    print("Sem dados para visualizar.")

### Análise Detalhada de Palavras-Chave
Nesta seção, vamos processar os títulos para extrair as palavras mais relevantes, removendo termos comuns (stopwords) e visualizando os resultados em tabela e gráfico.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import re

# Carregar dados
conn = sqlite3.connect(DATABASE_NAME)
df_analysis = pd.read_sql("SELECT title FROM articles", conn)
conn.close()

def get_keywords(titles):
    all_text = ' '.join(titles).lower()
    words = re.findall(r'\b[a-zà-ÿ]{4,}\b', all_text)
    stopwords_pt = {'para', 'com', 'uma', 'dos', 'das', 'pelo', 'pela', 'sobre', 'como', 'entre', 'está', 'será', 'nesta', 'seus', 'minha', 'mais', 'pelas', 'pelos', 'este', 'esta'}
    return [w for w in words if w not in stopwords_pt]

keywords = get_keywords(df_analysis['title'])
keyword_freq = pd.Series(keywords).value_counts().head(20).reset_index()
keyword_freq.columns = ['Palavra', 'Frequência']

# Gerar gráfico estático com Matplotlib (Garantia de visibilidade)
plt.figure(figsize=(10, 6))
sns.barplot(data=keyword_freq, x='Frequência', y='Palavra', palette='viridis')
plt.title('Top 20 Palavras mais Frequentes (Gráfico Estático)')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

### Gráfico de Dispersão das Datas de Publicação
Vamos converter as strings de data para objetos datetime e visualizar a linha do tempo das notícias.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
import sqlite3

# Carregar dados
conn = sqlite3.connect(DATABASE_NAME)
df_dates = pd.read_sql("SELECT title, date FROM articles", conn)
conn.close()

df_dates['date_dt'] = pd.to_datetime(df_dates['date'], dayfirst=True, errors='coerce')
df_dates = df_dates.dropna(subset=['date_dt']).sort_values('date_dt')

fig_scatter = px.scatter(
    df_dates,
    x='date_dt',
    y=[1]*len(df_dates),
    hover_data=['title'],
    title='Dispersão Temporal das Notícias Coletadas',
    labels={'date_dt': 'Data de Publicação', 'y': 'Artigo'},
    height=300
)

fig_scatter.update_layout(yaxis_visible=False, yaxis_showticklabels=False)

pio.show(fig_scatter, renderer="colab")